# Lab 1: Profiling LLM Inference Across Hardware

**Course**: *Dataflow Architectures for AI: From Principles to Practice*  
**Week**: 1, The Accelerator Landscape  
**Instructor**: Dr. Kaoutar El Maghraoui (IBM Research)  
**Lab Instructor**: Dr. Rashed Z. Bhatti (IBM Research)  
**Runtime**: Google Colab, GPU runtime required (T4)  
**Expected time**: 90–120 minutes

---

## Learning objectives
By the end of this lab you will be able to:
1. Build a repeatable benchmarking harness with torch.utils.benchmark for CPU and GPU inference.
2. Profile memory allocation during inference with torch.profiler and the CUDA memory stats.
3. Construct the first row of the Accelerator Landscape Map from measurements you made yourself.
4. Explain the CPU vs GPU gap in terms of SIMT parallelism, device-memory bandwidth, and precision.

## Prerequisites
Intermediate Python and PyTorch nn.Module. You have completed the Week 1 lecture on control-flow vs dataflow execution and the Accelerator Landscape Map.

## How this lab is graded

This lab includes **checkpoint cells for learner self-checks**: assertion cells inside each Part that verify your outputs land in the expected range as you work. They help you catch mistakes; they do **not** assign a grade.

Official course progress and certificate eligibility are tracked through the **Cognitive Class module review questions and the final exam**, not through this notebook. The analysis prompts in Part 5 are practice for the Module 1 review questions.

### Working through this lab
1. Run each cell in order. Make sure every **Checkpoint** cell passes (no `AssertionError`).
2. Fill in every `TODO` and answer every `YOUR ANALYSIS HERE` prompt.
3. When you are done, complete the **Module 1 review questions** on Cognitive Class.


## Environment Preflight

This cell detects your runtime environment and sets the execution mode:
- **Full GPU**: all exercises run on CUDA (recommended: T4 or better in Colab)
- **Reduced CPU**: GPU-dependent cells use CPU fallback with reduced workloads
- **Static read-only**: no execution; pre-computed outputs are displayed

In [ ]:
# === Environment Preflight ===
import sys
import importlib

# Check Python version
assert sys.version_info >= (3, 9), f"Python >= 3.9 required, got {sys.version}"

# Check PyTorch
try:
    import torch
    _torch_version = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
    assert _torch_version >= (2, 2), f"PyTorch >= 2.2 required, got {torch.__version__}"
except ImportError:
    raise RuntimeError("PyTorch not installed. Run: pip install torch>=2.2")

# Detect execution mode
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    MODE = "full-gpu"
    _device_name = torch.cuda.get_device_name(0)
else:
    DEVICE = torch.device("cpu")
    MODE = "reduced-cpu"
    _device_name = "CPU only"

print(f"╔══════════════════════════════════════════════════╗")
print(f"║  Environment Preflight                           ║")
print(f"╠══════════════════════════════════════════════════╣")
print(f"║  Python:    {sys.version.split()[0]:<37s}║")
print(f"║  PyTorch:   {torch.__version__:<37s}║")
print(f"║  Device:    {_device_name:<37s}║")
print(f"║  Mode:      {MODE:<37s}║")
print(f"╚══════════════════════════════════════════════════╝")

if MODE == "reduced-cpu":
    print("\n⚠️  No GPU detected. GPU-dependent cells will use CPU fallback.")
    print("   For full experience: Runtime → Change runtime type → T4 GPU")


## Part 0: Environment setup

We will use **Qwen 2.5 0.5B Instruct** throughout. It is small enough for Colab's free T4 GPU and representative enough of modern decoder-only LLM inference to make the measurements meaningful.

> **Why Qwen 2.5 0.5B?** It uses Grouped-Query Attention (GQA), RoPE positional embeddings, and SwiGLU; the three architectural choices that dominate modern open-weight LLMs. Whatever you learn here transfers to Llama 3, Mistral, Qwen 3, and the next model that comes out next quarter.


In [ ]:
# Install dependencies. Colab already has torch, we just need transformers and accelerate.
!pip install --upgrade pip
!pip install -q transformers accelerate


> **Roofline & Arithmetic-Intensity Conventions (this notebook)**
> - **Precision:** FP16 (IEEE half) unless otherwise noted
> - **FLOPs counting:** 1 fused multiply-add = 2 FLOPs
> - **Memory level:** Device DRAM (HBM/GDDR), not L2 or scratchpad
> - **Byte convention:** Bytes *transferred* between DRAM and compute (reads + writes)
> - **Sparsity:** Dense (no structural sparsity) unless stated
> - **Hardware SKU:** Detected at runtime via `torch.cuda.get_device_name()`; specs from `hardware_profiles.json`

In [ ]:
# PyTorch version compatibility check
import torch
print(f'PyTorch version: {torch.__version__}')
assert int(torch.__version__.split('.')[0]) >= 2, (
    f'This lab requires PyTorch 2.x or later. Got {torch.__version__}. '
    f'Run: pip install --upgrade torch'
)
print('✓ PyTorch version compatible')


In [ ]:
import os
import time
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.utils import benchmark as torch_bench
from torch.profiler import profile, record_function, ProfilerActivity

# Make outputs reproducible
torch.manual_seed(0)

# Quick sanity: GPU runtime is required for this lab.
assert torch.cuda.is_available(), (
    "GPU runtime required. In Colab: Runtime -> Change runtime type -> T4 GPU."
)

print(f"torch       = {torch.__version__}")
print(f"CUDA        = {torch.version.cuda}")
print(f"GPU         = {torch.cuda.get_device_name(0)}")
print(f"VRAM total  = {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

# Load tokenizer + model in fp16 (inference precision on most modern GPUs)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model_gpu = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).cuda().eval()
model_cpu = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).cpu().eval()

# Build a fixed input. Using a fixed input keeps the benchmark deterministic.
PROMPT = "Explain how a GPU differs from a dataflow accelerator in two sentences."
BATCH_SIZE = 1
SEQ_LEN = 64

inputs = tokenizer([PROMPT] * BATCH_SIZE, return_tensors="pt", padding="max_length",
                   truncation=True, max_length=SEQ_LEN)

inputs_gpu = {k: v.cuda() for k, v in inputs.items()}
inputs_cpu = {k: v.cpu() for k, v in inputs.items()}

# Print model size
n_params = sum(p.numel() for p in model_gpu.parameters())
print(f"Model: {MODEL_ID}")
print(f"Params: {n_params/1e6:.1f} M")
print(f"Input shape: {inputs['input_ids'].shape}")


## Part 1: Build a benchmarking harness

`torch.utils.benchmark.Timer` is the right tool for this: it handles GPU synchronization, warm-up, and variance estimation automatically. The naive `time.time()` around a forward pass will report *kernel launch time*, not *execution time*, because CUDA calls are asynchronous by default.

**Your task**: write a `bench_forward(model, inputs, label)` function that:
1. Warms up the model (run forward 3 times, discard results)
2. Uses `torch_bench.Timer` to measure a single forward pass
3. Returns the **median** latency in milliseconds

The `Timer` object's `.blocked_autorange()` method is what you want. It runs enough iterations to hit a stable measurement.


**💡 Hint: `bench_forward` (Part 1)**

<details>
<summary>Click to reveal a hint</summary>

- **Warm-up:** run the forward 3× inside `torch.inference_mode()`, then call `torch.cuda.synchronize()` if the model is on CUDA.
- **Timer:** `torch_bench.Timer(stmt="with torch.inference_mode(): model(**inputs)", globals={"model": model, "inputs": inputs, "torch": torch}, label=label, sub_label="forward")`
- **Return:** `m = timer.blocked_autorange(min_run_time=min_run_time)` then `return m.median * 1000.0` (seconds → ms).

</details>

#
#
#
 
M
e
a
s
u
r
e
m
e
n
t
 
B
e
s
t
 
P
r
a
c
t
i
c
e
s


>
 
*
*
⚠
️
 
S
i
n
g
l
e
-
r
u
n
 
t
i
m
i
n
g
s
 
c
a
n
 
b
e
 
m
i
s
l
e
a
d
i
n
g
.
*
*
 
G
P
U
 
p
e
r
f
o
r
m
a
n
c
e
 
v
a
r
i
e
s
 
d
u
e
 
t
o
 
t
h
e
r
m
a
l
 
t
h
r
o
t
t
l
i
n
g
,

>
 
b
a
c
k
g
r
o
u
n
d
 
p
r
o
c
e
s
s
e
s
,
 
a
n
d
 
m
e
m
o
r
y
 
s
t
a
t
e
.
 
A
l
w
a
y
s
 
u
s
e
 
m
u
l
t
i
p
l
e
 
m
e
a
s
u
r
e
m
e
n
t
s
.


*
*
P
r
o
t
o
c
o
l
 
f
o
r
 
r
e
l
i
a
b
l
e
 
b
e
n
c
h
m
a
r
k
s
:
*
*

1
.
 
*
*
W
a
r
m
u
p
:
*
*
 
3
-
5
 
i
t
e
r
a
t
i
o
n
s
 
(
d
i
s
c
a
r
d
e
d
)
 
t
o
 
f
i
l
l
 
c
a
c
h
e
s
 
a
n
d
 
t
r
i
g
g
e
r
 
J
I
T
 
c
o
m
p
i
l
a
t
i
o
n

2
.
 
*
*
R
e
p
e
a
t
:
*
*
 
N
 
≥
 
1
0
 
m
e
a
s
u
r
e
m
e
n
t
 
i
t
e
r
a
t
i
o
n
s

3
.
 
*
*
R
e
p
o
r
t
:
*
*
 
M
e
d
i
a
n
 
(
r
o
b
u
s
t
 
t
o
 
o
u
t
l
i
e
r
s
)
 
±
 
I
Q
R
 
o
r
 
p
e
r
c
e
n
t
i
l
e
s
 
(
P
5
,
 
P
5
0
,
 
P
9
5
)

4
.
 
*
*
S
y
n
c
h
r
o
n
i
z
e
:
*
*
 
A
l
w
a
y
s
 
c
a
l
l
 
`
t
o
r
c
h
.
c
u
d
a
.
s
y
n
c
h
r
o
n
i
z
e
(
)
`
 
b
e
f
o
r
e
 
t
i
m
i
n
g
 
G
P
U
 
o
p
s

5
.
 
*
*
I
s
o
l
a
t
e
:
*
*
 
U
s
e
 
`
t
o
r
c
h
.
i
n
f
e
r
e
n
c
e
_
m
o
d
e
(
)
`
 
o
r
 
`
t
o
r
c
h
.
n
o
_
g
r
a
d
(
)
`
 
t
o
 
a
v
o
i
d
 
a
u
t
o
g
r
a
d
 
o
v
e
r
h
e
a
d



In [ ]:
#
 
T
O
D
O
:
 
I
m
p
l
e
m
e
n
t
 
a
 
r
o
b
u
s
t
 
b
e
n
c
h
m
a
r
k
i
n
g
 
u
t
i
l
i
t
y

#
 
R
e
q
u
i
r
e
m
e
n
t
s
:

#
 
1
.
 
A
c
c
e
p
t
 
a
 
c
a
l
l
a
b
l
e
 
f
n
 
a
n
d
 
r
u
n
 
i
t
 
n
_
w
a
r
m
u
p
 
t
i
m
e
s
 
(
d
i
s
c
a
r
d
 
r
e
s
u
l
t
s
)

#
 
2
.
 
R
u
n
 
f
n
 
n
_
m
e
a
s
u
r
e
 
t
i
m
e
s
,
 
r
e
c
o
r
d
i
n
g
 
w
a
l
l
-
c
l
o
c
k
 
t
i
m
e
 
f
o
r
 
e
a
c
h

#
 
3
.
 
R
e
t
u
r
n
 
s
t
a
t
i
s
t
i
c
s
:
 
m
e
d
i
a
n
,
 
m
e
a
n
,
 
s
t
d
,
 
P
5
,
 
P
9
5
,
 
I
Q
R

#
 
4
.
 
H
a
n
d
l
e
 
C
U
D
A
 
s
y
n
c
h
r
o
n
i
z
a
t
i
o
n
 
(
t
o
r
c
h
.
c
u
d
a
.
s
y
n
c
h
r
o
n
i
z
e
(
)
 
b
e
f
o
r
e
/
a
f
t
e
r
)

#

#
 
H
i
n
t
:
 
U
s
e
 
t
i
m
e
.
p
e
r
f
_
c
o
u
n
t
e
r
(
)
 
f
o
r
 
h
i
g
h
-
r
e
s
o
l
u
t
i
o
n
 
t
i
m
i
n
g

#
 
H
i
n
t
:
 
n
p
.
p
e
r
c
e
n
t
i
l
e
(
t
i
m
e
s
,
 
[
5
,
 
5
0
,
 
9
5
]
)
 
g
i
v
e
s
 
y
o
u
 
P
5
,
 
m
e
d
i
a
n
,
 
P
9
5


i
m
p
o
r
t
 
t
i
m
e

i
m
p
o
r
t
 
n
u
m
p
y
 
a
s
 
n
p


d
e
f
 
b
e
n
c
h
_
r
o
b
u
s
t
(
f
n
,
 
n
_
w
a
r
m
u
p
=
5
,
 
n
_
m
e
a
s
u
r
e
=
2
0
,
 
s
y
n
c
_
c
u
d
a
=
T
r
u
e
)
:

 
 
 
 
"
"
"
R
u
n
 
f
n
(
)
 
w
i
t
h
 
w
a
r
m
u
p
,
 
r
e
t
u
r
n
 
t
i
m
i
n
g
 
s
t
a
t
i
s
t
i
c
s
 
i
n
 
m
i
l
l
i
s
e
c
o
n
d
s
.
"
"
"

 
 
 
 
#
 
Y
O
U
R
 
C
O
D
E
 
H
E
R
E

 
 
 
 
p
a
s
s


#
 
T
e
s
t
 
y
o
u
r
 
i
m
p
l
e
m
e
n
t
a
t
i
o
n
:

#
 
x
 
=
 
t
o
r
c
h
.
r
a
n
d
n
(
1
0
2
4
,
 
1
0
2
4
,
 
d
e
v
i
c
e
=
D
E
V
I
C
E
)

#
 
s
t
a
t
s
 
=
 
b
e
n
c
h
_
r
o
b
u
s
t
(
l
a
m
b
d
a
:
 
t
o
r
c
h
.
m
m
(
x
,
 
x
)
)

#
 
a
s
s
e
r
t
 
'
m
e
d
i
a
n
_
m
s
'
 
i
n
 
s
t
a
t
s

#
 
a
s
s
e
r
t
 
s
t
a
t
s
[
'
n
'
]
 
=
=
 
2
0



In [ ]:
def bench_forward(model, inputs, label, min_run_time=1.0):
    """Measure median forward-pass latency (ms) using torch.utils.benchmark.

    Args:
        model: the nn.Module to benchmark (must already be in eval mode and on-device).
        inputs: dict of input tensors matching model signature (on the right device).
        label: string label for the measurement (used for display).
        min_run_time: minimum seconds of measurement (Timer will iterate until this is met).

    Returns:
        median_latency_ms: float, the measured median forward-pass latency in ms.
    """
    # TODO 1.1: warm up the model with 3 forward passes. Use torch.inference_mode().
    #             (inference_mode is the right context for inference benchmarking: like
    #             no_grad it disables autograd, but it also skips extra version/view
    #             bookkeeping, so it is a little cleaner and lower-overhead.)
    #             Why warm up? The first few forwards trigger lazy initialization,
    #             kernel compilation, and cache warming, none of which we want to measure.
    pass  # REPLACE

    # TODO 1.2: build a torch_bench.Timer.
    #             The stmt should run the forward inside inference_mode, i.e.
    #             stmt="with torch.inference_mode(): model(**inputs)".
    #             Pass globals={"model": model, "inputs": inputs, "torch": torch} so the
    #             Timer can see these names. Pass label=label and sub_label="forward".
    timer = None  # REPLACE

    # TODO 1.3: run timer.blocked_autorange(min_run_time=min_run_time) and return
    #             the median in milliseconds.
    #             Note: the returned Measurement has `.median` in seconds.
    raise NotImplementedError("Remove this once you have implemented the function")

In [ ]:
# --- Checkpoint 1 -------------------------------------------------------
# Validates your bench_forward() function on a trivial model.
class _TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(128, 128)
    def forward(self, input):
        return self.fc(input)

_check_model  = _TinyModel().cuda().eval()
_check_inputs = {"input": torch.randn(4, 128, device="cuda")}

_ms = bench_forward(_check_model, _check_inputs, "tiny-check", min_run_time=0.3)
assert _ms is not None, "bench_forward returned None; did you return the median?"
assert 0.001 < _ms < 50.0, f"Reference linear bench looks off: {_ms} ms"
print(f"[OK] bench_forward works. Reference tiny model: {_ms:.3f} ms")


## Part 2: Run inference on CPU and GPU

Now use your harness to measure the same forward pass on CPU and GPU.

**Your task**:
1. Call `bench_forward` on `model_cpu` with `inputs_cpu`.
2. Call `bench_forward` on `model_gpu` with `inputs_gpu`.
3. Compute **tokens per second** for each: `throughput = (batch_size * seq_len) / (latency_ms / 1000)`.
4. Compute the **speedup** (GPU throughput / CPU throughput).

Record the numbers in the dict `perf`.

> **What does this tokens/sec measure?** We call `model(**inputs)` once, so this is the throughput of a **single forward pass (prefill) over a fixed prompt**: how fast the model ingests the input tokens. It is *not* generated-output tokens/sec (autoregressive decode throughput), which measures how fast new tokens are produced one at a time. We look at decode throughput in a later week.


In [ ]:
perf = {}

# TODO 2.1: measure CPU latency and throughput
#             (CPU inference is slow, we pass min_run_time=0.5 to keep the lab short)
#             Note: this throughput is forward-pass / prefill tokens/sec over a fixed
#             prompt, NOT generated-output tokens/sec.
cpu_ms = None  # REPLACE
cpu_tps = None  # REPLACE: tokens per second (prefill)

# TODO 2.2: measure GPU latency and throughput
gpu_ms = None  # REPLACE
gpu_tps = None  # REPLACE

# TODO 2.3: compute speedup as GPU_tps / CPU_tps
speedup = None  # REPLACE

perf["cpu_ms"] = cpu_ms
perf["gpu_ms"] = gpu_ms
perf["cpu_tps"] = cpu_tps
perf["gpu_tps"] = gpu_tps
perf["speedup"] = speedup

print(f"CPU:    {cpu_ms:7.1f} ms/fwd   {cpu_tps:7.1f} tok/s")
print(f"GPU:    {gpu_ms:7.1f} ms/fwd   {gpu_tps:7.1f} tok/s")
print(f"Speedup (GPU / CPU): {speedup:.1f}x")
print("(tok/s = forward-pass / prefill throughput over a fixed prompt, not generated-output tok/s)")

In [ ]:
# --- Checkpoint 2 -------------------------------------------------------
# Sanity checks on your measurements. Ranges below are deliberately wide because
# Colab hardware varies. We just want to catch obvious mistakes.
assert perf["cpu_ms"] > 50,    f"CPU latency suspiciously low: {perf['cpu_ms']} ms"
assert perf["gpu_ms"] < 200,   f"GPU latency suspiciously high: {perf['gpu_ms']} ms"
assert perf["cpu_ms"] > perf["gpu_ms"], "CPU should be slower than GPU here."
assert perf["speedup"] > 1.5,  f"Expected GPU speedup > 1.5x on this workload, got {perf['speedup']:.1f}x"
print(f"[OK] Measurements look reasonable. Speedup = {perf['speedup']:.1f}x")

## Part 2b: Arithmetic Intensity Preview (Seeds Week 6)

Now that you have measured throughput, let's compute the **arithmetic intensity** (AI) of your forward pass.
This tells you whether your workload is **compute-bound** or **memory-bound** on this GPU.

**Arithmetic Intensity** = FLOPs performed / Bytes transferred from memory

For a decoder-only transformer forward pass (rough estimate):
- FLOPs ≈ 2 × n_params × batch_size × seq_len (each parameter does one multiply-add per input token)
- Bytes ≈ n_params × 2 (read all weights in FP16 once per forward pass, at batch=1)

The **ridge point** of your GPU is: Peak TFLOPS / Peak Memory BW (TB/s).
If AI < ridge → memory-bound. If AI > ridge → compute-bound.

In [ ]:
# Part 2b: Compute arithmetic intensity of your forward pass

# Hardware specs, we provide a lookup table so your code works on different GPUs.
# Source: NVIDIA datasheets (T4: 2018, A100: 2020, L4: 2023, V100: 2017)
HW_SPECS = {
    'T4':   {'fp16_tflops': 65.0,  'bw_tb_s': 0.320},
    'A100': {'fp16_tflops': 312.0, 'bw_tb_s': 2.039},
    'L4':   {'fp16_tflops': 121.0, 'bw_tb_s': 0.300},
    'V100': {'fp16_tflops': 125.0, 'bw_tb_s': 0.900},
}

def lookup_hw(gpu_name):
    for key, specs in HW_SPECS.items():
        if key in gpu_name:
            return key, specs
    return 'T4', HW_SPECS['T4']

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'T4 (assumed)'
hw_key, hw = lookup_hw(gpu_name)
GPU_PEAK_TFLOPS_FP16 = hw['fp16_tflops']
GPU_PEAK_BW_TBS = hw['bw_tb_s']
print(f"Using {hw_key} specs: {GPU_PEAK_TFLOPS_FP16} TFLOPS FP16, {GPU_PEAK_BW_TBS} TB/s")

# TODO 2b.1: Calculate the approximate FLOPs for one forward pass
#   Formula: 2 * n_params * BATCH_SIZE * SEQ_LEN
#   (This counts one multiply + one add per parameter per token)
approx_flops = None  # REPLACE

# TODO 2b.2: Calculate the bytes of model weights read from device memory
#   Formula: n_params * 2 (FP16 = 2 bytes per parameter)
#   Note: at batch=1, activations are small compared to weights
weight_bytes = None  # REPLACE

# TODO 2b.3: Compute arithmetic intensity
ai = None  # REPLACE: approx_flops / weight_bytes

# Compute the GPU's ridge point
ridge_point = GPU_PEAK_TFLOPS_FP16 * 1e12 / (GPU_PEAK_BW_TBS * 1e12)  # FLOPs/byte

print(f"Approximate FLOPs per forward: {approx_flops/1e9:.1f} GFLOPs")
print(f"Weight bytes (FP16):           {weight_bytes/1e9:.2f} GB")
print(f"Arithmetic Intensity:          {ai:.1f} FLOPs/byte")
print(f"GPU Ridge Point ({hw_key}):       {ridge_point:.0f} FLOPs/byte")
print()
if ai < ridge_point:
    print(f"→ AI ({ai:.1f}) < Ridge ({ridge_point:.0f}): Your workload is MEMORY-BOUND.")
    print(f"  The GPU's tensor cores are {(1 - ai/ridge_point)*100:.0f}% underutilized!")
    print(f"  This is typical for small-batch LLM inference. Week 6 will explore this deeply.")
else:
    print(f"→ AI ({ai:.1f}) > Ridge ({ridge_point:.0f}): Your workload is COMPUTE-BOUND.")
    print(f"  The batch size is large enough to amortize weight reads.")


## Part 3: Profile memory allocation

Latency is half the story. On accelerators, *where and when data moves* is usually the other half. For this part you'll:

1. Measure **peak GPU memory** during a forward pass using `torch.cuda.max_memory_allocated`.
2. Use `torch.profiler` with `profile_memory=True` to capture the memory timeline of one forward pass.
3. Identify the top-3 operators by CUDA time.

This is also your first look at how a **cache-based** memory hierarchy (GPU) behaves. In later labs you'll contrast this with **software-managed** scratchpads (TPU, Spyre).


**💡 Hint: peak memory (Part 3.1 / 3.2)**

<details>
<summary>Click to reveal a hint</summary>

```python
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
with torch.inference_mode():
    _ = model_gpu(**inputs_gpu)
torch.cuda.synchronize()
peak_bytes = torch.cuda.max_memory_allocated()

weights_bytes = sum(p.numel() * p.element_size() for p in model_gpu.parameters())
mem_overhead_ratio = peak_bytes / weights_bytes
```

</details>

In [ ]:
# TODO 3.1: reset peak memory counter, run a forward pass, record peak.
#             Functions you need: torch.cuda.reset_peak_memory_stats(),
#                                 torch.cuda.max_memory_allocated().
#             Run the forward inside torch.inference_mode(), and remember to
#             torch.cuda.synchronize() after the forward.
peak_bytes = None  # REPLACE

print(f"Peak GPU memory for a single forward: {peak_bytes / 1e6:.1f} MB")

# TODO 3.2: compute the "memory overhead ratio": peak_bytes divided by the total
#             size of the model weights.
#             Compute the weight bytes from the actual parameter dtypes (robust even
#             if the model dtype changes later):
#                 weights_bytes = sum(p.numel() * p.element_size()
#                                     for p in model_gpu.parameters())
mem_overhead_ratio = None  # REPLACE
print(f"Memory overhead ratio (peak / weights): {mem_overhead_ratio:.2f}x")

In [ ]:
# Use torch.profiler to capture the op-level memory timeline.
# Note: profile_memory adds overhead, we are not trying to measure latency here.
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    profile_memory=True,
    record_shapes=True,
) as prof:
    with torch.inference_mode():
        with record_function("llm_forward"):
            _ = model_gpu(**inputs_gpu)
    torch.cuda.synchronize()

# Print top 10 operators by CUDA time.
print(prof.key_averages().table(
    sort_by="cuda_time_total",
    row_limit=10,
))


**💡 Hint: top-3 operators (Part 3.3)**

<details>
<summary>Click to reveal a hint</summary>

```python
events = [e for e in prof.key_averages() if e.key != "llm_forward"]
events.sort(key=lambda e: e.self_cuda_time_total, reverse=True)
top3 = [(e.key, e.self_cuda_time_total) for e in events[:3]]
```

Remember: `llm_forward` is the `record_function` label we added, not a real operator. Filter it out so the true dominant op shows up.

</details>

In [ ]:
# TODO 3.3: extract the top-3 operators by *self* CUDA time from the profiler.
#             prof.key_averages() returns a list of FunctionEventAvg objects.
#             Each has .key (op name) and .self_cuda_time_total (microseconds).
#             IMPORTANT: filter out the "llm_forward" entry first. That is just the
#             record_function wrapper label we added, not a real model operator. If you
#             leave it in it will dominate the list and hide the true top operator.
#             Sort the remaining ops descending, take top 3, store as (name, time_us) tuples.
top3 = None  # REPLACE  -- should be List[Tuple[str, float]]

for name, t_us in top3:
    print(f"  {name:40s}  {t_us/1000:8.2f} ms")

In [ ]:
# --- Checkpoint 3 -------------------------------------------------------
_weights_bytes = sum(p.numel() * p.element_size() for p in model_gpu.parameters())
assert peak_bytes > _weights_bytes, "Peak memory should exceed just the model weights."
assert mem_overhead_ratio >= 1.0, f"Overhead ratio should be >= 1.0, got {mem_overhead_ratio}"
assert len(top3) == 3, "top3 must contain exactly 3 operators."
assert all(isinstance(t, tuple) and len(t) == 2 for t in top3), "top3 entries must be (name, time_us) tuples."
assert all(name != "llm_forward" for name, _ in top3), "The 'llm_forward' wrapper label must be filtered out of top3."
print("[OK] Memory profile captured, top-3 operators extracted.")

## Part 4: Build your first row of the Accelerator Landscape Map

The course's **Accelerator Landscape Map** is not vendor marketing, it's a table you build from measurement. In this lab you populate the *Compute & Memory* row for **GPU (NVIDIA T4 or whatever Colab gave you)** from the numbers you collected.

Fill in the `landscape_row` dict below using your measurements from Parts 1–3.


In [ ]:
# TODO 4.1: fill in the Landscape Map row for the GPU you measured.
landscape_row = {
    "accelerator":          None,  # e.g. (f"GPU ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "CPU")
    "execution_model":      None,  # one short phrase, e.g. "SIMT / control-flow"
    "memory_hierarchy":     None,  # one short phrase, e.g. "L1/L2/GDDR6 global memory (hw-managed cache)"
    "programming_model":    None,  # one short phrase, e.g. "CUDA / PyTorch eager"
    "forward_latency_ms":   None,  # from perf["gpu_ms"]
    "throughput_tps":       None,  # from perf["gpu_tps"]
    "peak_memory_MB":       None,  # from peak_bytes / 1e6
    "dominant_op":          None,  # top3[0][0]
}

for k, v in landscape_row.items():
    print(f"  {k:22s}: {v}")


In [ ]:
# --- Checkpoint 4 -------------------------------------------------------
required_keys = {
    "accelerator", "execution_model", "memory_hierarchy", "programming_model",
    "forward_latency_ms", "throughput_tps", "peak_memory_MB", "dominant_op",
}
assert set(landscape_row.keys()) == required_keys, "Missing or extra keys in landscape_row."
assert all(v is not None for v in landscape_row.values()), "All landscape_row fields must be filled."
assert isinstance(landscape_row["forward_latency_ms"], (int, float))
assert isinstance(landscape_row["throughput_tps"],    (int, float))
print("[OK] Landscape Map row complete. You have contributed the first row!")


## Part 5: Written analysis

These prompts are practice for the Module 1 review questions. Be specific and cite the numbers *you* measured; that is exactly the reasoning the review questions will check. Write 3–6 sentences per answer; vague or slogan-level answers miss the point of the exercise.

---

### 5.1: Explain the measured GPU speedup over CPU in terms of *specific* architectural features.

You measured roughly a `[YOUR_SPEEDUP]x` speedup. Name at least three architectural features that contribute to this gap and explain the contribution of each. (SIMT, GPU device-memory bandwidth, cache hierarchy, thousands of parallel threads, dedicated matmul units, low-precision arithmetic; pick three and *explain* them.)

**YOUR ANALYSIS HERE**

---

### 5.2: Why is the peak memory larger than just the model weights?

Your overhead ratio was roughly `[YOUR_RATIO]x`. Where does the extra memory go during a forward pass? Be specific about *which* tensors are live simultaneously for a 0.5B-parameter decoder-only transformer with GQA.

**YOUR ANALYSIS HERE**

---

### 5.3: Comparative prediction (the Landscape Map question).

Predict how the **memory overhead ratio** would likely differ on a **dataflow accelerator with software-managed scratchpads** (e.g., Spyre, TPU) vs the cache-based GPU you just measured. Would it be higher, lower, roughly the same? Justify with reference to how scratchpads manage live tensors.

You will revisit this prediction in Week 4. It's okay to be wrong, but be *concrete* about why you're predicting what you're predicting.

**YOUR ANALYSIS HERE**

In [ ]:
# --- Save your results (optional, for your own records) -----------------
# Cognitive Class tracks your official progress through the module review questions
# and the final exam; there is no file to upload from this notebook. This cell just
# saves your measurements locally so you can refer back to them (e.g. when you fill in
# the Landscape Map or answer the review questions).
import json as _json
my_results = {
    "perf":               perf,
    "peak_bytes":         peak_bytes,
    "mem_overhead_ratio": mem_overhead_ratio,
    "top3":               [(n, t) for (n, t) in top3],
    "landscape_row":      landscape_row,
}
with open("lab1_results.json", "w") as _f:
    _json.dump(my_results, _f, indent=2, default=str)
print("Saved lab1_results.json for your own reference.")
print("Official progress: complete the Module 1 review questions on Cognitive Class.")

---

## Optional: Dataflow Deep Dive (Spyre hardware access required)

> *Only complete this section if you have been granted Spyre access through the IBM Research academic partnership.*

Open `../optional_spyre/lab1_spyre_extension.ipynb` and run the same benchmarking harness against Spyre hardware. Extend your `landscape_row` into a second row for Spyre, and write a **3rd analysis paragraph** comparing where Spyre's static dataflow model produces materially different numbers from your GPU run.

---

*Course: Dataflow Architectures for AI, Week 1 Lab, v1.0 · © 2026 Dr. Kaoutar El Maghraoui*


> **Spyre vocabulary (consistent across all labs):** `import torch_sendnn` registers the `aiu` device; compile with `torch.compile(model, backend="sendnn")`; dump the backend IR with `TORCH_COMPILE_DEBUG=1` (look for `sdsc_*.json`); raise verbosity with `SENDNN_LOG_LEVEL=DEBUG`. Requires Spyre access via an IBM Research partnership.
